# Benchmark: Windowed vs Full-Grid 3D Barrier Solver

Compares `iterative_3d_barrier(..., windowed=True)` against
`iterative_3d_barrier(..., windowed=False)` on multiple downsampled
versions of the real Elastix volume.

**Windowed mode** operates on padded bounding boxes around each
connected component of negative Jdet voxels with a frozen boundary
ring. Memory is bounded by the largest window, so even the full
`(528, 320, 456)` volume is tractable on CPU.

**Full-grid mode** optimises the entire field at once. L-BFGS-B
stores ~20 correction pairs of size `3 × D × H × W × 8 bytes`, so the
full-resolution volume requires roughly **37 GB** of working memory
and is skipped above `FULLGRID_MAX_DOFS`.

Metrics reported per (factor, mode) pair: neg-Jdet count, min Jdet,
L2 distortion, wall time, peak RSS.

## Table of Contents

1. [Configuration](#Configuration)
2. [Memory-sampling helper](#Memory-sampling-helper)
3. [Run benchmark grid](#Run-benchmark-grid)
4. [Results table](#Results-table)
5. [Plots](#Plots)

In [1]:
import os
import sys
import time
import threading

import numpy as np
import matplotlib.pyplot as plt
import psutil

# benchmark_utils.py lives in benchmarks/ (two levels up from this notebook)
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

from dvfopt import jacobian_det3D, scale_dvf_3d
from dvfopt.core.iterative3d_barrier import iterative_3d_barrier
from benchmark_utils import (
    get_output_dir, save_results_csv, save_summary_json,
    log_run_header, log_run_footer, results_to_rows,
    show_and_save, reset_figure_counter,
)

In [2]:
METHOD = "barrier"
NOTEBOOK_NAME = "windowed-vs-fullgrid"
OUTPUT_DIR = get_output_dir(METHOD, NOTEBOOK_NAME, base="../../output")
reset_figure_counter()
summary = log_run_header(METHOD, NOTEBOOK_NAME, OUTPUT_DIR)

  Benchmark  : windowed-vs-fullgrid
  Method     : barrier
  Timestamp  : 2026-04-15T23:38:27
  Output dir : ..\..\output\barrier\windowed-vs-fullgrid


## Configuration

In [3]:
DOWNSAMPLE_FACTORS = [1 / 4, 1 / 2, 1]   # quarter, half, full
MODES = [("windowed", True), ("full-grid", False)]

# Skip full-grid above this DOF cap (3 * D * H * W).
# L-BFGS-B stores ~20 correction vectors of 8 bytes per DOF, so
# 60M DOFs ~= 10 GB working memory.
FULLGRID_MAX_DOFS = 60_000_000

MAX_MINIMIZE_ITER = 200
FULL_VOLUME_PATH = "../../../data/corrected_correspondences_count_touching/registered_output/deformation3d.npy"

results = []   # list of dict rows

print(f"Downsample factors : {DOWNSAMPLE_FACTORS}")
print(f"Modes              : {[name for name, _ in MODES]}")
print(f"Full-grid DOF cap  : {FULLGRID_MAX_DOFS:,}")

Downsample factors : [0.25, 0.5, 1]
Modes              : ['windowed', 'full-grid']
Full-grid DOF cap  : 60,000,000


## Memory-sampling helper

Runs `func` in the foreground while a background thread samples the
process RSS every `poll_s` seconds. Returns `(result, elapsed, peak_rss, baseline_rss)`
in MB.

In [4]:
def _peak_rss_during(func, *args, poll_s=0.1, **kwargs):
    proc = psutil.Process()
    baseline = proc.memory_info().rss
    peak = baseline
    stop = threading.Event()

    def _sampler():
        nonlocal peak
        while not stop.is_set():
            try:
                rss = proc.memory_info().rss
            except psutil.Error:
                break
            if rss > peak:
                peak = rss
            stop.wait(poll_s)

    t = threading.Thread(target=_sampler, daemon=True)
    t.start()
    t0 = time.time()
    try:
        result = func(*args, **kwargs)
    finally:
        elapsed = time.time() - t0
        stop.set()
        t.join()
    mb = 1024 * 1024
    return result, elapsed, peak / mb, baseline / mb

## Run benchmark grid

In [ ]:
if not os.path.exists(FULL_VOLUME_PATH):
    raise FileNotFoundError(f"Full volume file not found: {FULL_VOLUME_PATH}")

full_dvf = np.load(FULL_VOLUME_PATH)
_, Df, Hf, Wf = full_dvf.shape
print(f"Original full volume: {full_dvf.shape}", flush=True)

header = f"{'Factor':>6s}  {'Mode':>10s}  {'Shape':>18s}  {'DOFs':>12s}  {'Neg init':>9s}  {'Neg final':>10s}  {'Min Jdet':>12s}  {'L2':>10s}  {'Time (s)':>10s}  {'Peak MB':>10s}  {'Status':>8s}"
print(header)
print("-" * len(header))

for factor in DOWNSAMPLE_FACTORS:
    factor_label = {1 / 4: "1/4", 1 / 2: "1/2", 1: "1/1"}.get(factor, f"{factor:.3f}")
    print()
    print("=" * len(header), flush=True)
    print(f"  Downsample factor: {factor_label}", flush=True)
    print("=" * len(header), flush=True)
    if factor == 1:
        ds_dvf = full_dvf.copy()
    else:
        new_shape = (
            max(1, int(round(Df * factor))),
            max(1, int(round(Hf * factor))),
            max(1, int(round(Wf * factor))),
        )
        ds_dvf = scale_dvf_3d(full_dvf, new_shape)

    D, H, W = ds_dvf.shape[1:]
    n_dofs = 3 * D * H * W
    shape_label = f"{D}x{H}x{W}"

    jac_init = jacobian_det3D(ds_dvf)
    init_neg = int((jac_init <= 0).sum())
    init_min = float(jac_init.min())

    for mode_name, windowed in MODES:
        print("-" * len(header), flush=True)
        row = {
            "factor": factor_label,
            "mode": mode_name,
            "shape": shape_label,
            "n_dofs": n_dofs,
            "n_neg_init": init_neg,
            "min_jdet_init": init_min,
        }

        if not windowed and n_dofs > FULLGRID_MAX_DOFS:
            row.update({
                "n_neg_final": None, "min_jdet": None, "l2_err": None,
                "time": None, "peak_mb": None, "status": "SKIPPED",
            })
            results.append(row)
            print(f"{factor_label:>6s}  {mode_name:>10s}  {shape_label:>18s}  "
                  f"{n_dofs:>12,}  {init_neg:>9d}  {'-':>10s}  {'-':>12s}  "
                  f"{'-':>10s}  {'-':>10s}  {'-':>10s}  {'SKIPPED':>8s}", flush=True)
            continue

        print(f"[{factor_label} / {mode_name}] Running on {shape_label} "
              f"(init neg={init_neg}, min={init_min:+.3f}, DOFs={n_dofs:,}) ...",
              flush=True)

        try:
            (phi, elapsed, peak_mb, baseline_mb) = _peak_rss_during(
                iterative_3d_barrier, ds_dvf,
                verbose=1,
                max_minimize_iter=MAX_MINIMIZE_ITER,
                windowed=windowed,
            )
            jac_final = jacobian_det3D(phi)
            n_neg_final = int((jac_final <= 0).sum())
            min_jdet_final = float(jac_final.min())
            l2_err = float(np.linalg.norm(phi - ds_dvf))
            status = "OK" if n_neg_final == 0 else "PARTIAL"
            row.update({
                "n_neg_final": n_neg_final,
                "min_jdet": min_jdet_final,
                "l2_err": l2_err,
                "time": elapsed,
                "peak_mb": peak_mb,
                "status": status,
            })
            print(f"{factor_label:>6s}  {mode_name:>10s}  {shape_label:>18s}  "
                  f"{n_dofs:>12,}  {init_neg:>9d}  {n_neg_final:>10d}  "
                  f"{min_jdet_final:>+12.6f}  {l2_err:>10.4f}  "
                  f"{elapsed:>10.1f}  {peak_mb:>10.1f}  [{status}]", flush=True)
        except MemoryError as e:
            row.update({
                "n_neg_final": None, "min_jdet": None, "l2_err": None,
                "time": None, "peak_mb": None, "status": "OOM",
            })
            print(f"  [OOM] {factor_label} / {mode_name}: {e}", flush=True)
        except Exception as e:
            row.update({
                "n_neg_final": None, "min_jdet": None, "l2_err": None,
                "time": None, "peak_mb": None, "status": f"ERROR",
            })
            print(f"  [ERROR] {factor_label} / {mode_name}: {type(e).__name__}: {e}",
                  flush=True)

        results.append(row)

del full_dvf

## Results table

In [ ]:
import json

print(json.dumps(results, indent=2, default=str))

## Plots

Side-by-side bars per downsample factor for windowed (blue) vs
full-grid (red). Skipped / OOM / errored runs are omitted.

In [ ]:
COLOR_WINDOWED = "#1f77b4"
COLOR_FULLGRID = "#d62728"

def _plot_metric(metric, ylabel, title, logy=False):
    factors = [r["factor"] for r in results]
    factor_labels = []
    for f in factors:
        if f not in factor_labels:
            factor_labels.append(f)

    windowed_vals = []
    fullgrid_vals = []
    for f in factor_labels:
        w = next((r[metric] for r in results
                  if r["factor"] == f and r["mode"] == "windowed"
                  and r[metric] is not None), np.nan)
        g = next((r[metric] for r in results
                  if r["factor"] == f and r["mode"] == "full-grid"
                  and r[metric] is not None), np.nan)
        windowed_vals.append(w)
        fullgrid_vals.append(g)

    x = np.arange(len(factor_labels))
    width = 0.35

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(x - width / 2, windowed_vals, width, label="windowed",
           color=COLOR_WINDOWED)
    ax.bar(x + width / 2, fullgrid_vals, width, label="full-grid",
           color=COLOR_FULLGRID)

    for i, v in enumerate(windowed_vals):
        if np.isfinite(v):
            ax.text(x[i] - width / 2, v, f"{v:.1f}", ha="center",
                    va="bottom", fontsize=8)
    for i, v in enumerate(fullgrid_vals):
        if np.isfinite(v):
            ax.text(x[i] + width / 2, v, f"{v:.1f}", ha="center",
                    va="bottom", fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(factor_labels)
    ax.set_xlabel("Downsample factor")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    if logy:
        ax.set_yscale("log")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    show_and_save(OUTPUT_DIR)
    return fig

_plot_metric("time", "Wall time (s)",
             "Wall time: windowed vs full-grid", logy=True)
_plot_metric("peak_mb", "Peak RSS (MB)",
             "Peak memory: windowed vs full-grid", logy=True)
_plot_metric("l2_err", "L2 distortion",
             "L2 distortion: windowed vs full-grid")

In [ ]:
# --- Save results ---
columns = ["factor", "mode", "shape", "n_dofs", "n_neg_init",
           "min_jdet_init", "n_neg_final", "min_jdet", "l2_err",
           "time", "peak_mb", "status"]
rows = []
for r in results:
    row = {}
    for c in columns:
        val = r.get(c)
        if isinstance(val, float):
            val = round(val, 6)
        row[c] = val
    rows.append(row)
save_results_csv(rows, columns, OUTPUT_DIR)

summary["results"] = rows
summary["total_cases"] = len(rows)
summary["converged"] = sum(1 for r in rows if r["status"] == "OK")
save_summary_json(summary, OUTPUT_DIR)